# Worked example: which borrowers does a proactive call help stay current?

Companion to [guide chapter 9.1](../guide/09-other-uses.md). **All data is synthetic** (`payment_assistance_data.py`).

**Scenario.** Over two quarters, 800 small-business borrowers of a community lender miss a payment. **Every borrower
gets the standard help**: a reminder letter and a notice of hardship options. Loan officers also have time for a
limited number of **proactive calls** offering to talk through a short-term payment plan. To learn whom the call helps,
the lender randomized which borrowers got a call. That tests the *extra* help, and no one was denied the standard response.

**Outcome:** the loan is brought current (cured) within 60 days of the missed payment.

**Question:** next quarter the team can make about 120 calls. Who should get them?

In [1]:
import numpy as np
import pandas as pd
from payment_assistance_data import generate, next_quarter
from uplift_tools import lift_with_ci, segment_lift, cost_per_incremental, sample_size_per_group

pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
df = generate()
df.groupby("called")["cured_60d"].agg(["size", "mean"])

,size,mean
called,,
0,388,0.503
1,412,0.546


## 1. Was the test big enough?

With a cure rate near 50% without a call, detecting a 10-point improvement needs:

In [2]:
sample_size_per_group(p_control=0.50, lift=0.10)

{'holdout': 385, 'contacted': 385, 'total': 770}

**Reading it:** about 800 borrowers split 50/50 is enough to detect a 10-point overall improvement, but not small
differences between segments. That is why this example uses a handful of segments defined in advance, not a model
(chapter 5).

## 2. Does the call help at all?

In [3]:
r = lift_with_ci(df, "called", "cured_60d")
{k: round(v, 3) if isinstance(v, float) else v for k, v in r.items() if k in
 ["n_contacted", "n_holdout", "rate_contacted", "rate_holdout", "lift", "ci_low", "ci_high", "p_value"]}

{'n_contacted': 412,
 'n_holdout': 388,
 'rate_contacted': np.float64(0.546),
 'rate_holdout': np.float64(0.503),
 'lift': np.float64(0.044),
 'ci_low': np.float64(-0.026),
 'ci_high': np.float64(0.113),
 'p_value': np.float64(0.217)}

**Reading it:** borrowers who got a call were cured about 4 points more often (55% vs 50%), but the interval runs from
about −3 to +11 points, so on average the effect can't be distinguished from zero. The test was sized to detect a 10-point
average effect, and the true average turned out smaller.

Stopping here would lead to the conclusion that calls don't help. But an average can hide a large effect in one group and
none in the others. That is what the segment table checks.

## 3. Which borrowers does it help?

In [4]:
seg = segment_lift(df, "segment", "called", "cured_60d").sort_values("lift_shrunk", ascending=False)
seg

,segment,prospects,rate_holdout,rate_contacted,lift,ci_low,ci_high,lift_shrunk
2,"First missed payment, seasonal business",165,0.533,0.808,0.276,0.139,0.412,0.217
0,Already on a hardship plan,92,0.154,0.264,0.110,-0.054,0.274,0.088
1,"First missed payment, other",322,0.652,0.665,0.013,-0.091,0.117,0.018
3,Repeat late (2+ missed in 12 months),221,0.382,0.345,-0.038,-0.165,0.089,-0.019


**Reading it:**

* **First missed payment, seasonal business**: a large, clear effect, about +28 points, with an interval well above
  zero. These borrowers usually have income coming, and a conversation about timing keeps them current.
* **Already on a hardship plan**: the point estimate is positive, but with only 92 borrowers the interval is wide and
  includes zero. Shrinkage pulls it down. Treat it as **unproven** and keep testing.
* **First missed payment, other** and **repeat late**: no measurable effect from the extra call. Repeat-late borrowers are
  among the *highest-risk*, but a proactive call is not what changes their outcome. They may need a different kind of help
  (restructuring, counselling), which would be its own test.

This is the point chapter 9 makes: **the borrowers most at risk are not necessarily the ones a call helps most.** An
early-warning list (who is at risk) and an uplift list (who the call helps) are different lists.

## 4. Allocating next quarter's calls

In [5]:
nq = next_quarter()
lift = seg.set_index("segment")["lift_shrunk"]
nq["expected_lift"] = nq["segment"].map(lift)
CALLS = 120
rng = np.random.default_rng(5)

policies = {
    "By measured lift (segment table)": nq["expected_lift"].values + rng.random(len(nq)) * 1e-6,
    "Highest risk first (repeat late, hardship plan)": nq["segment"].map({
        "Already on a hardship plan": 4, "Repeat late (2+ missed in 12 months)": 3,
        "First missed payment, other": 2, "First missed payment, seasonal business": 1}).values + rng.random(len(nq)) * 1e-3,
    "Random": rng.random(len(nq)),
}
rows = []
for name, s in policies.items():
    chosen = np.argsort(-s)[:CALLS]
    rows.append({"policy": name, "extra loans cured (true, simulation only)": nq["true_effect"].values[chosen].sum()})
pd.DataFrame(rows).set_index("policy").round(1)

,"extra loans cured (true, simulation only)"
policy,
By measured lift (segment table),18.400
"Highest risk first (repeat late, hardship plan)",2.500
Random,9.200


**Reading it:** the same 120 calls cure about twice as many loans as random calling, and about seven times as many as
calling the highest-risk borrowers first, when they go to the segment the test showed they help. Only the simulation knows the true effect. In practice, you use
the segment table and keep testing.

## 5. Keep learning

* **Keep randomizing a share of calls** (say 20%) within the prioritized segments, so you notice if the effect changes.
* **Test different help for the high-risk segments.** For example, a restructuring offer versus the standard notice.
  Again, everyone gets help; the test compares types.
* **Record it** with the [test plan](../templates/test-plan-template.md) and
  [results](../templates/campaign-results-template.md) templates.